# PosgenPy Full Analysis Example (Python-only)

This is a python-only version of `PosgenPy_Full_Analysis_Example.ipynb`. It does
**not** use posgen or libatomprobe at all - the cluster search (DBSCAN + envelope +
erosion, matching posgen's "maximum separation" algorithm), the relabelling/
randomisation control runs, and all the corrections/metrics are done by the
`posgenpy` package in this repo, using only numpy/pandas/scikit-learn/scipy.

It follows the same overall workflow as the original notebook:
1. Order (kNN) sweep, at fixed dmax
2. dmax sweep, at fixed order
3. Pick final (dmax, order, n_min)
4. Final cluster search + corrections (edge-cluster removal, Fm deconvolution, metrics)
5. Composition/violin/3D plots
6. CV01 parameter-sensitivity check
7. Save a one-row summary CSV

It runs on the small bundled test dataset (`posgenpyTest_easy.pos` +
`examples/posgenpyTest.rrng`) so it can be run top-to-bottom as-is; swap the
file paths/ion lists for your own data.


In [ ]:
######################### INPUT REQUIRED #########################
import posgenpy as pgp
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

sample_name = "PosgenPyTest"
pos_file_path = "posgenpyTest_easy.pos"
range_file_path = "examples/posgenpyTest.rrng"
output_folder = "pygenpy_output_pythononly/"

core_ions = ["Cu", "Ni"]
bulk_ions = ["Fe"]

random_runs = 20
confidence_threshold = 0.95
detector_efficiency = 0.52       # LEAP5000+ ~0.52, LEAP3000 ~0.37
atomic_density_per_nm3 = 85.49   # BCC-iron


## Load data

In [ ]:
pos_df = pgp.read_pos_to_df(pos_file_path)
rrng_df = pgp.read_rrng(range_file_path)

# AP Suite exports don't use the "Range..." line format read_rrng expects -
# fall back to the AP-Suite reader if nothing was parsed.
if rrng_df.empty:
    rrng_df = pgp.read_apt_range_file(range_file_path)

ion_colors = pgp.get_ion_colours_from_rrng(rrng_df)

print(f"{len(pos_df):,} atoms")
rrng_df


## Order (kNN) sweep

Sweep `order` (k) at a fixed `dmax`, running a real cluster search plus
`random_runs` relabelled (mc-shuffled) control searches at each value. The
KDTree over all atom positions is built once and reused across every swept
value and every relabel run.


In [ ]:
######################### INPUT REQUIRED #########################
order_values = [2, 3, 4, 5, 6, 7]
fixed_dmax = 0.5

order_manifest = pgp.run_parameter_sweep(
    pos_df, rrng_df, core_ions, bulk_ions,
    swept_param="order", swept_values=order_values,
    fixed_dmax=fixed_dmax, n_min=2, n_relabel_runs=random_runs,
    output_dir=output_folder, prefix=sample_name,
    verbose=False,
)


### Plotting real-vs-random cluster counts

In [ ]:
n_min_values = list(range(2, 21, 2))

pgp.plot_real_clusters_across_swept_param(order_manifest, "Order", n_min_values)
pgp.plot_real_cluster_ratio_across_swept_param(
    order_manifest, "Order", n_min_values, threshold=confidence_threshold
)


### Find optimal N_min per Order value

In [ ]:
optimal_n_min_order, real_clusters_order, random_clusters_order = pgp.optimise_n_min(
    order_manifest, n_min_values=range(1, 200), confidence_threshold=confidence_threshold
)
optimal_n_min_order


### Cluster composition vs Order

In [ ]:
pgp.plot_cluster_composition_across_swept_param_absolute(
    order_manifest, core_ions, optimal_n_min_order, "Order", ion_colors=ion_colors
)
pgp.plot_cluster_composition_across_swept_param_relative(
    order_manifest, "Order", core_ions, optimal_n_min_order, ion_colors=ion_colors
)


## dmax sweep

Same as above, sweeping `dmax` at a fixed `order`.


In [ ]:
######################### INPUT REQUIRED #########################
dmax_values = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
fixed_order = 5

dmax_manifest = pgp.run_parameter_sweep(
    pos_df, rrng_df, core_ions, bulk_ions,
    swept_param="dmax", swept_values=dmax_values,
    fixed_order=fixed_order, n_min=2, n_relabel_runs=random_runs,
    output_dir=output_folder, prefix=sample_name,
    verbose=False,
)


In [ ]:
pgp.plot_real_clusters_across_swept_param(dmax_manifest, "dmax", n_min_values)
pgp.plot_real_cluster_ratio_across_swept_param(
    dmax_manifest, "dmax", n_min_values, threshold=confidence_threshold
)


In [ ]:
optimal_n_min_dmax, real_clusters_dmax, random_clusters_dmax = pgp.optimise_n_min(
    dmax_manifest, n_min_values=range(1, 200), confidence_threshold=confidence_threshold
)
optimal_n_min_dmax


In [ ]:
pgp.plot_cluster_composition_across_swept_param_absolute(
    dmax_manifest, core_ions, optimal_n_min_dmax, "dmax", ion_colors=ion_colors
)
pgp.plot_cluster_composition_across_swept_param_relative(
    dmax_manifest, "dmax", core_ions, optimal_n_min_dmax, ion_colors=ion_colors
)


## Select final parameters

Pick `(dmax, order, n_min)` from the sweep results above.


In [ ]:
######################### INPUT REQUIRED #########################
selected_dmax = 0.5
selected_order = 5
selected_n_min = optimal_n_min_dmax[dmax_values.index(selected_dmax)]
selected_n_min


## Final real cluster search + corrections

Run the cluster search once more at the selected parameters (with plenty of
relabel runs for the final composition/metrics), then apply the corrections
pipeline: edge-cluster detection, Fm-peak deconvolution (if applicable), and
per-tip metrics.


In [ ]:
tree, tree_positions = pgp.build_position_tree(pos_df)

final_manifest_entry = pgp.run_relabel_sweep(
    pos_df, rrng_df, core_ions, bulk_ions,
    dmax=selected_dmax, k=selected_order, n_min=2,
    envelope_radius=selected_dmax / 2, d_erosion=selected_dmax / 2,
    n_relabel_runs=random_runs, run_name=f"{sample_name}_final",
    output_dir=output_folder, tree=tree, tree_positions=tree_positions,
)

clustered_atoms = pd.read_parquet(final_manifest_entry["real_clustered_atoms_path"])
cluster_stats = pd.read_parquet(final_manifest_entry["real_cluster_stats_path"])

ranged_atoms = clustered_atoms.loc[clustered_atoms["ion"].notna()]
cluster_stats = pgp.remove_edge_clusters(
    ranged_atoms, clustered_atoms, cluster_stats, random_state=0, print_log=True
)
cluster_stats = cluster_stats.loc[cluster_stats["size"] >= selected_n_min].reset_index(drop=True)

unclustered_stats = pgp.summarise_unclustered(clustered_atoms)

cluster_stats.to_csv(f"{output_folder}{sample_name}_cluster-stats_corrected.csv", index=False)
unclustered_stats.to_csv(f"{output_folder}{sample_name}_unclustered-stats_corrected.csv", index=False)

cluster_stats.head()


## Composition plots (ternary + stacked bar)

In [ ]:
ternary_ions = [ion for ion in core_ions + bulk_ions if ion in cluster_stats.columns]

fig = px.scatter_ternary(
    cluster_stats, a=ternary_ions[0], b=ternary_ions[1], c=ternary_ions[2],
    size="size", color="size",
    title=f"{sample_name}: cluster composition (ternary)",
)
fig.show()


In [ ]:
bar_df = cluster_stats.sort_values("size")
fig = go.Figure()
for ion in ternary_ions:
    fig.add_trace(go.Bar(x=bar_df["cluster_id"].astype(str), y=bar_df[ion], name=ion))
fig.update_layout(barmode="stack", title=f"{sample_name}: per-cluster composition", xaxis_title="Cluster ID")
fig.show()


## Cluster-size / kNN cluster-distance distributions across the dmax sweep

In [ ]:
violin_frames = []
for entry, n_min in zip(dmax_manifest, optimal_n_min_dmax):
    stats = pd.read_parquet(entry["real_cluster_stats_path"])
    stats = stats.loc[stats["size"] >= n_min]
    stats = pgp.find_knn_cluster_distance(stats)
    stats["dmax"] = entry["swept_value"]
    violin_frames.append(stats)

violin_df = pd.concat(violin_frames, ignore_index=True)

px.violin(violin_df, x="dmax", y="size", box=True, points="all",
          title="Cluster size distribution vs dmax").show()
px.violin(violin_df, x="dmax", y="closest_cluster_distance", box=True, points="all",
          title="Nearest-neighbour cluster distance vs dmax").show()


## 3D visualisation of selected clusters

In [ ]:
largest_clusters = cluster_stats.sort_values("size", ascending=False)["cluster_id"].head(3).tolist()

pgp.show_clusters_in_pos(
    final_manifest_entry["real_clustered_atoms_path"], largest_clusters, ion_colors=ion_colors
)


In [ ]:
pgp.compare_selected_clusters_across_param(
    dmax_manifest, "dmax", chosen_value=selected_dmax, selected_clusters=largest_clusters
)


## CV01 sensitivity

Perturb the selected (dmax, order) by (+-0.05 nm, +-1) and re-run the final
pipeline twice more, recording the max absolute deviation per metric as an
error estimate.


In [ ]:
dmax_interval = 0.05
order_interval = 1

cv01_metrics = []
for sign in (-1, 1):
    this_dmax = round(selected_dmax + sign * dmax_interval, 2)
    this_order = selected_order + sign * order_interval

    entry = pgp.run_relabel_sweep(
        pos_df, rrng_df, core_ions, bulk_ions,
        dmax=this_dmax, k=this_order, n_min=2,
        envelope_radius=this_dmax / 2, d_erosion=this_dmax / 2,
        n_relabel_runs=random_runs, run_name=f"{sample_name}_cv01_{sign}",
        output_dir=output_folder, tree=tree, tree_positions=tree_positions,
    )

    cv_atoms = pd.read_parquet(entry["real_clustered_atoms_path"])
    cv_stats = pd.read_parquet(entry["real_cluster_stats_path"])
    cv_ranged = cv_atoms.loc[cv_atoms["ion"].notna()]
    cv_stats = pgp.remove_edge_clusters(cv_ranged, cv_atoms, cv_stats, random_state=0)
    cv_stats = cv_stats.loc[cv_stats["size"] >= selected_n_min]
    cv_unclustered = pgp.summarise_unclustered(cv_atoms)

    metrics = pgp.get_cluster_metrics(
        reconstruction_name=f"{sample_name}_cv01_{sign}", reconstruction_location=output_folder,
        sample_type=sample_name, cluster_stats_df=cv_stats, unclustered_stats_df=cv_unclustered,
        core_ions=core_ions, bulk_ions=bulk_ions, dmax=this_dmax, order=this_order,
        n_min=selected_n_min, detector_efficiency=detector_efficiency,
        atomic_density_per_nm3=atomic_density_per_nm3, cluster_analysis_type="cv01",
    )
    cv01_metrics.append(metrics)

main_unclustered = pgp.summarise_unclustered(clustered_atoms)
main_metrics = pgp.get_cluster_metrics(
    reconstruction_name=sample_name, reconstruction_location=output_folder,
    sample_type=sample_name, cluster_stats_df=cluster_stats, unclustered_stats_df=main_unclustered,
    core_ions=core_ions, bulk_ions=bulk_ions, dmax=selected_dmax, order=selected_order,
    n_min=selected_n_min, detector_efficiency=detector_efficiency,
    atomic_density_per_nm3=atomic_density_per_nm3, cluster_analysis_type="main",
)

cluster_analysis_summary = pd.DataFrame([main_metrics])
cv01_df = pd.DataFrame(cv01_metrics)

cols_for_cv01 = [
    "volume fraction", "number density per nm3", "clustered volume nm3",
    "number of all clusters", "number of non-edge clusters", "number of edge clusters",
]
for col in cols_for_cv01:
    cluster_analysis_summary[f"{col} cv01 error"] = (
        cv01_df[col] - main_metrics[col]
    ).abs().max()

cluster_analysis_summary.T


## Save summary + light disk cleanup

In [ ]:
cluster_analysis_summary.to_csv(
    f"{output_folder}{sample_name}_cluster_analysis_summary.csv", index=False
)

# Unlike the posgen-driven workflow, there's no per-relabel .pos file
# explosion to prune - only the per-swept-value real_clustered_atoms.parquet
# files are worth dropping once you're done exploring a sweep.
import glob, os

keep = {final_manifest_entry["real_clustered_atoms_path"]}
sweep_atom_files = glob.glob(f"{output_folder}{sample_name}_order_*_real_clustered_atoms.parquet")
sweep_atom_files += glob.glob(f"{output_folder}{sample_name}_dmax_*_real_clustered_atoms.parquet")
for path in sweep_atom_files:
    if path not in keep:
        os.remove(path)

print("Done.")


---
Thanks for using PosgenPy!
